# Feature consistency audit — zero classifier fits

B3 recovery and independent comparison are complete. This next check measures
how often specified feature relationships fail **before and after the unchanged
historical transformation** on the verified 119,571-row prepared dataset.
It does not change your original results or fit a model.

1. Select **Runtime → Run all** in Google Colab.
2. Upload the **updated PQC_IDS_Reproduction_Working_Copy.zip supplied with this notebook**.
3. Connect the Drive account containing the original four CSV files.
4. Return **PQC_IDS_Feature_Consistency_Results.zip**, even if it reports an error.

The default CSV folder is `/content/drive/MyDrive/cicids2018`. Edit `DATA_DIR`
near the start of the code cell only if that folder moved. The single code cell
performs setup, auditing and export together. Re-running it starts a separate
complete audit. A result ZIP is also saved under `MyDrive/pqc_ids_feature_audits`.
The declared Python and package versions are restored in a separate interpreter;
setup and loading the CSV files may take time even though no models are trained.

## What is checked

All four archived footprints are applied separately to the same prepared matrix.
Fifteen diagnostic rules cover directional/overall maxima versus byte totals,
means between minima and maxima, directional mean/total relationships, and
positive rates when the corresponding numerator and duration are positive.

Counts distinguish **pre-existing, introduced and resolved violations**, with
separate counts where a rule becomes evaluable only after transformation.
Every count has an evaluable denominator, and is reported by class and by the
original bidirectional eligibility rule. Primary comparisons allow an absolute
rounding tolerance of `1e-5` plus a relative tolerance of `1e-6`; exact-comparison
sensitivity counts are also saved. These do not change reproduction tolerances.
The zero-original-byte rate branch has its own count table.

The script checks the original source, input-file and prepared-matrix hashes,
and confirms that ineligible rows and the operator's untouched columns stay
unchanged. Compact row masks and deterministically chosen examples permit
independent recounting. Raw CSV files are excluded from the export.

## Interpretation

`audit_completed` means the checks ran; it does **not** mean the approximation
is physically valid. Packet-length comparisons assume the corresponding fields
use the same byte basis. Directional mean equality is explicitly an extractor
semantics diagnostic. Rate checks use only the sign of duration, with no unit
conversion. This is not a packet-trace/TLS validation or a performance experiment.
No replacement transformation is introduced here. Its effect on classifier
scores would require a separately specified comparison.

The notebook and audit were prepared with assistant assistance. Original
experiment source files remain unchanged.

In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import shutil
import stat
import subprocess
import sys
import tempfile
import traceback
import uuid
import zipfile
from google.colab import files, drive

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
SESSION = Path(tempfile.mkdtemp(prefix="pqc_feature_audit_", dir="/content"))

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


# Edit this folder only if the four original CSV files moved.
DATA_DIR = Path("/content/drive/MyDrive/cicids2018")
DRIVE_OUTPUT = SESSION / "records"
DRIVE_OUTPUT.mkdir()
RUN_OUTPUT = DRIVE_OUTPUT / "results"
SUMMARY = {"run_id": RUN_ID, "status": "preparing", "classifier_fits": 0,
           "scope": "Feature consistency audit of the unchanged historical operator",
           "historical_results_modified": False, "colab_kernel_python": platform.python_version()}
DRIVE_CONNECTED = False

def save_summary():
    (DRIVE_OUTPUT / "notebook_status.json").write_text(json.dumps(SUMMARY, indent=2) + "\n")

save_summary()
try:
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("Upload exactly one updated working-copy ZIP.")
    archive_path = SESSION / "working_copy.zip"
    archive_path.write_bytes(uploaded[zip_names[0]])
    del uploaded
    extracted = SESSION / "package"
    with zipfile.ZipFile(archive_path) as archive:
        if archive.testzip() is not None:
            raise ValueError("The ZIP failed its integrity check. Download the working copy again.")
        for info in archive.infolist():
            path = PurePosixPath(info.filename)
            if path.is_absolute() or ".." in path.parts or stat.S_ISLNK(info.external_attr >> 16):
                raise ValueError("Unexpected archive path: " + info.filename)
        archive.extractall(extracted)
    manifest_paths = list(extracted.glob("*/package_manifest.json"))
    if len(manifest_paths) != 1:
        raise ValueError("Expected one working-copy package manifest.")
    PACKAGE = manifest_paths[0].parent
    manifest = json.loads(manifest_paths[0].read_text())
    listed = set()
    for item in manifest["files"]:
        path = (PACKAGE / item["path"]).resolve()
        if not path.is_relative_to(PACKAGE.resolve()):
            raise ValueError("Invalid manifest path.")
        if not path.is_file() or path.stat().st_size != item["bytes"] or sha256(path) != item["sha256"]:
            raise ValueError("Package file does not match the manifest: " + item["path"])
        listed.add(item["path"])
    actual = {p.relative_to(PACKAGE).as_posix() for p in PACKAGE.rglob("*") if p.is_file()}
    if actual != listed | {"package_manifest.json"} or len(listed) != len(manifest["files"]):
        raise ValueError("The archive contains unlisted files or duplicate manifest records.")
    if not (PACKAGE / "scripts/audit_feature_consistency.py").is_file():
        raise ValueError("This is an older working copy. Use the ZIP provided with this notebook.")
    print("Working-copy files verified:", len(listed))
    print("Run identifier:", RUN_ID)

    SUMMARY["package_sha256"] = sha256(archive_path)
    SUMMARY["package_manifest_sha256"] = sha256(PACKAGE / "package_manifest.json")
    shutil.copy2(PACKAGE / "package_manifest.json", DRIVE_OUTPUT / "working_copy_manifest.json")
    for name in ("main_known_versions.json", "main_requirements.txt", "feature_consistency_plan.json"):
        shutil.copy2(PACKAGE / "environment" / name, DRIVE_OUTPUT / name)
    TARGET = json.loads((PACKAGE / "environment/main_known_versions.json").read_text())
    drive.mount("/content/drive")
    DRIVE_CONNECTED = True
    SUMMARY["data_directory"] = str(DATA_DIR)
    SUMMARY["target"] = TARGET
    save_summary()

    RUN_ENV = os.environ.copy()
    for name in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV"):
        RUN_ENV.pop(name, None)
    RUN_ENV["PYTHONNOUSERSITE"] = "1"
    RUN_ENV["UV_PYTHON_INSTALL_DIR"] = str(SESSION / "python_runtimes")
    RUN_ENV["UV_CACHE_DIR"] = str(SESSION / "uv_cache")

    def run_logged(command, log_name, env):
        with (DRIVE_OUTPUT / log_name).open("w", encoding="utf-8") as log:
            log.write("Command: " + json.dumps([str(s) for s in command]) + "\n")
            log.flush()
            with subprocess.Popen([str(s) for s in command], stdout=subprocess.PIPE,
                                  stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                                  text=True, bufsize=1, env=env) as process:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
                return process.wait()

    ENVIRONMENT_READY = False
    VENV = SESSION / "feature_audit_environment"
    FIT_PYTHON = VENV / "bin/python"
    try:
        tooling = SESSION / "uv_tooling"
        bootstrap = [sys.executable, "-m", "pip", "install", "--target", tooling,
                     "--no-deps", "uv==0.12.15"]
        if run_logged(bootstrap, "01_uv_install.log", RUN_ENV) != 0:
            raise RuntimeError("Could not install the pinned uv tool.")
        uv_env = RUN_ENV.copy()
        uv_env["PYTHONPATH"] = str(tooling)
        uv = [sys.executable, "-m", "uv"]
        create = uv + ["venv", "--python", TARGET["python"], "--managed-python", VENV]
        if run_logged(create, "02_python_setup.log", uv_env) != 0:
            raise RuntimeError("Could not create the requested Python runtime.")
        install = uv + ["pip", "install", "--python", FIT_PYTHON, "--only-binary", ":all:",
                        "-r", PACKAGE / "environment/main_requirements.txt"]
        if run_logged(install, "03_packages_install.log", uv_env) != 0:
            raise RuntimeError("Could not install the declared package versions.")
        ENVIRONMENT_READY = True
        SUMMARY["status"] = "environment_installed"
    except Exception as error:
        SUMMARY["status"] = "environment_setup_failed"
        SUMMARY["error"] = str(error)
        (DRIVE_OUTPUT / "setup_traceback.txt").write_text(traceback.format_exc())
        print("Setup failed:", error)
    finally:
        save_summary()
    print("Environment ready:", ENVIRONMENT_READY)

    if not ENVIRONMENT_READY:
        raise RuntimeError("Audit did not run because environment setup failed; return the exported logs.")
    SUMMARY["status"] = "running_audit"
    save_summary()
    returncode = run_logged([FIT_PYTHON, "-u", PACKAGE / "scripts/audit_feature_consistency.py",
                             "--data-dir", DATA_DIR, "--output", RUN_OUTPUT,
                             "--cache-dir", SESSION / "input_cache"],
                            "04_feature_consistency_audit.log", RUN_ENV)
    SUMMARY["runner_returncode"] = returncode
    status_path = RUN_OUTPUT / "audit_status.json"
    if not status_path.is_file():
        raise RuntimeError("Audit runner produced no status record; return the ZIP.")
    audit_status = json.loads(status_path.read_text())
    SUMMARY["audit_status"] = audit_status["status"]
    if returncode != 0 or audit_status["status"] != "audit_completed" or audit_status["classifier_fits"] != 0:
        raise RuntimeError("Audit did not complete; return the ZIP containing its failure evidence.")
    SUMMARY["status"] = "audit_completed"
    print(json.dumps(audit_status["counts"], indent=2))
    print("Checks completed. This status does not mean every feature relationship passed.")

except BaseException as error:
    SUMMARY["status"] = "audit_incomplete"
    SUMMARY["error"] = str(error)
    (DRIVE_OUTPUT / "notebook_traceback.txt").write_text(traceback.format_exc())
    print("Please return the ZIP with this recorded issue:", error)
finally:
    SUMMARY["exported_utc"] = datetime.now(timezone.utc).isoformat()
    save_summary()
    result_files = sorted(p for p in DRIVE_OUTPUT.rglob("*") if p.is_file())
    items = [{"path": p.relative_to(DRIVE_OUTPUT).as_posix(), "bytes": p.stat().st_size,
              "sha256": sha256(p)} for p in result_files]
    manifest_path = DRIVE_OUTPUT / "result_manifest.json"
    manifest_path.write_text(json.dumps({"run_id": RUN_ID, "files": items}, indent=2) + "\n")
    result_files.append(manifest_path)
    result_zip = SESSION / "PQC_IDS_Feature_Consistency_Results.zip"
    with zipfile.ZipFile(result_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in result_files:
            archive.write(path, path.relative_to(DRIVE_OUTPUT).as_posix())
    with zipfile.ZipFile(result_zip) as archive:
        if archive.testzip() is not None:
            raise RuntimeError("Result archive failed its integrity check.")
    if DRIVE_CONNECTED:
        try:
            saved = Path("/content/drive/MyDrive/pqc_ids_feature_audits") / RUN_ID
            saved.mkdir(parents=True, exist_ok=False)
            shutil.copy2(result_zip, saved / result_zip.name)
            print("Drive copy:", saved / result_zip.name)
        except OSError as error:
            print("Drive copy failed, but the downloadable ZIP is available:", error)
    print("Final status:", SUMMARY["status"])
    print("Classifier fits: 0")
    print("ZIP SHA-256:", sha256(result_zip))
    files.download(str(result_zip))
